In [ ]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from tqdm import tqdm

class Reader:
    def __init__(self, examples_path, labels_path=None):
        self.examples_path = examples_path
        self.read_file()
        if labels_path is not None:
            self.labels = np.load(labels_path)

    def read_file(self):
        # În loc de citire texte linie cu linie, citim direct tensorii de imagini (sau patch-uri)
        # Presupunem format .npy: train_images va fi de forma (Nr_Imagini, H, W)
        self.data = np.load(self.examples_path)

# Citim datele respectând fix arhitectura clasei Reader din soluția oficială
train_reader = Reader("./data_set/train_images.npy", "./data_set/train_labels.npy")
test_reader = Reader("./data_set/test_images.npy", "./data_set/test_labels.npy")
patches_reader = Reader("./data_set/patch_filters.npy", None)

# ######### Ex2 (Adaptat) - Extragere Patch Frequencies ##########
def convolve(sample, kernels):
    # kernels: (b, kernel_h, kernel_w)
    # sample: (initial_h, initial_w)
    b, kernel_h, kernel_w = kernels.shape
    initial_h, initial_w = sample.shape

    output_h = initial_h - kernel_h + 1
    output_w = initial_w - kernel_w + 1

    if output_h < 0 or output_w < 0:
        return np.zeros((b, 1))

    # Aplatizăm patch-urile pentru a face produsul scalar ca în soluția oficială
    kernels_flat = kernels.reshape(b, -1)
    norm_kernels = np.linalg.norm(kernels_flat, axis=1)
    norm_kernels[norm_kernels == 0] = 1e-10 # Evităm împărțirea la zero

    output = np.zeros((b, output_h * output_w))
    idx = 0
    for i in range(output_h):
        for j in range(output_w):
            # Extragem sub-fereastra curentă din imagine
            window = sample[i:i+kernel_h, j:j+kernel_w].flatten()
            norm_window = np.linalg.norm(window)
            if norm_window == 0:
                norm_window = 1e-10

            # Formula exactă din rezolvarea 2025, dar aplicată pe window aplatizat
            output[:, idx] = np.sum(window[None, :] * kernels_flat, axis=1) / (norm_window * norm_kernels)
            idx += 1
    return output

def apply_convolution(data, filters, threshold):
    new_data = []
    print("Aplicare convoluție 2D pe imagini...")
    for example in tqdm(data):
        convolution_output = convolve(example, np.array(filters))
        new_data.append(np.sum(convolution_output > threshold, axis=1))
    return np.array(new_data)

# Aplicăm operația pentru a obține frecvența patch-urilor
train_data = apply_convolution(train_reader.data, patches_reader.data, threshold=0.9)
test_data = apply_convolution(test_reader.data, patches_reader.data, threshold=0.9)

####### Ex1 (KNN) ###########
# Adaptat la indicația că s-a cerut KNN
print("Ex1 (KNN):")
knn = KNeighborsClassifier(n_neighbors=5, metric='manhattan')
knn.fit(train_data, train_reader.labels)
preds = knn.predict(test_data)
print(f"Accuracy: {np.mean(preds == test_reader.labels)}")

#### Ex2 (KRR) ####
print("Ex2 (KRR):")
predictions = []
# Presupunem 4 clase ca în problema originală cu telefoanele (sau texte)
for i in range(4):
    new_train_labels = ((train_reader.labels == i) * 2) - 1
    krr = KernelRidge(kernel="linear", alpha=100)
    krr.fit(train_data, new_train_labels)
    predictions.append(krr.predict(test_data))

predictions = np.array(predictions)
predictions = np.argmax(predictions, axis=0)
print(f"Accuracy KRR: {(predictions == test_reader.labels).mean()}")

#### Ex3 (SVM) ####
print("Ex3 (SVM):")

def hellinger_kernel(features1, features2):
    features1 = np.sqrt(features1)
    features2 = np.sqrt(features2)
    matrix = np.matmul(features1, features2.T)
    return matrix

train_matrix = hellinger_kernel(train_data, train_data)
test_matrix = hellinger_kernel(test_data, train_data)

model = SVC(kernel='precomputed', C=10)
model.fit(train_matrix, train_reader.labels)
preds_svm = model.predict(test_matrix)
print(f"Accuracy SVM: {np.mean(preds_svm == test_reader.labels)}")